# Cài đặt RAG

## Cài đặt

In [ ]:
!pip install langchain-qdrant
!pip install accelerate bitsandbytes
!pip install bert_score
!pip install xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121
!pip install optimum qdrant-client wikipedia FastAPI uvicorn pyngrok
!pip install --upgrade pydantic
!pip install vllm
!pip install ragas
!pip install -U \
      python-dotenv \
      langchain \
      langchain_openai \
      langchain_community \
      langchain-huggingface \
      langchain-google-genai \
      streamlit \
      faiss-cpu \
      sentence-transformers \
      pypdf \
      docx2txt\
      vllm\
      triton
!pip install autoawq
!pip install accelerate transformers

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached triton-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.1/264.1 MB 34.5 MB/s eta 0:00:00
  Attempting uninstall: compressed-tensors
    Found existing installation: compressed-tensors 0.8.1
    Uninstalling compressed-tensors-0.8.1:
      Successfully uninstalled compressed-tensors-0.8.1
  Attempting uninstall: vllm
    Found existing installation: vllm 0.6.6.post1
    Uninstalling vllm-0.6.6.post1:
      Successfully uninstalled vllm-0.6.6.post1


## Khởi tạo biến môi trường

In [ ]:
#Khai báo khóa API key
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"]  = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Cài đặt thông tin truy cập vector database
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")
EMBEDDINGS_MODEL_NAME="intfloat/multilingual-e5-base"
HUGGINGFACE_API_KEY= userdata.get("HUGGINGFACE_API_KEY")
QDRANT_COLLECTION_NAME = "ITUS_e5_600"
GENERATE_MODEL_NAME="AITeamVN/Vi-Qwen2-3B-RAG"

## RAG gemini

In [ ]:
!pip install pyvi sentence-transformers nltk rouge-score

In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [18]:
from langchain.schema.document import Document
from langchain.schema.retriever import BaseRetriever
from langchain.retrievers import WikipediaRetriever
from langchain_qdrant import QdrantVectorStore
from langchain.llms import HuggingFacePipeline
from qdrant_client import QdrantClient
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.chains import RetrievalQA, MultiRetrievalQAChain
from langchain.llms import VLLM
from langchain.llms import HuggingFaceHub
from typing import List
import asyncio
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from asyncio import to_thread
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from pyvi import ViTokenizer
from nltk.translate.meteor_score import meteor_score
from sentence_transformers import SentenceTransformer, util
import time

class LLMServe:
    def __init__(self, model_name=GENERATE_MODEL_NAME) -> None:
        self.embeddings = self.load_embeddings()
        self.current_source = "qdrant"
        self.retriever = self.load_retriever(retriever_name=self.current_source, embeddings=self.embeddings)
        self.pipe = self.load_model_pipeline(model_name=model_name, max_new_tokens=512)
        self.prompt = self.load_prompt_template()
        self.sbert_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        self.rag_pipeline = self.load_rag_pipeline(
            llm=self.pipe,
            retriever=self.retriever,
            prompt=self.prompt
        )

    def load_embeddings(self):
        embeddings = HuggingFaceInferenceAPIEmbeddings(
            model_name=EMBEDDINGS_MODEL_NAME,
            api_key=HUGGINGFACE_API_KEY,
        )
        return embeddings

    def load_retriever(self, retriever_name, embeddings):
        """
        Load and create appropriate retriever based on retriever_name
        """
        base_retriever = None
        if retriever_name == "wiki":
            base_retriever = WikipediaRetriever(
                lang="vi",
                doc_content_chars_max=800,
                top_k_results=10
            )
        elif retriever_name == "qdrant":
            client = QdrantClient(
                url=QDRANT_URL,
                api_key=QDRANT_API_KEY,
                prefer_grpc=False
            )

            db = QdrantVectorStore(
                client=client,
                embedding=embeddings,
                collection_name=QDRANT_COLLECTION_NAME
            )
            base_retriever = db.as_retriever(search_kwargs={"k": 10})

        return base_retriever

    def load_model_pipeline(self, model_name="gemini", max_new_tokens=512):
        """
        Load and create appropriate model pipeline based on model_name
        """
        if model_name == "gemini":
            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
        else:
            llm = VLLM(
                model=model_name,
                trust_remote_code=True,
                max_new_tokens=max_new_tokens,
                top_k=10,
                top_p=0.95,
                temperature=0.4,
                dtype="half",
            )
        return llm

    def load_prompt_template(self):
        query_template = """
                    <|system|>
                    '''Thực hiện trả lời câu hỏi từ thông tin có trong ngữ cảnh được cho. Chú ý các yêu cầu sau:
                    - Câu trả lời phải chính xác, đầy đủ và có liên quan đến câu hỏi.
                    - Chỉ sử dụng các thông tin có trong ngữ cảnh được cung cấp.
                    - Nếu có nhiều ngữ cảnh liên quan với câu hỏi thì kết hợp các ngữ cảnh để tổng hợp thông tin.
                    - Nếu ngữ cảnh không chứa thông tin về câu trả lời thì từ chối trả lời và không suy luận gì thêm và cung cấp thông tin liên lạc của khoa như sau để người dùng liên lạc:
                    "Vui lòng liên lạc Khoa Công Nghệ Thông Tin, trường Đại học Khoa Học Tự Nhiên - Đại học Quốc Gia TP.Hồ Chí Minh để giải đáp:
                    Địa chỉ: Phòng I.54, toà nhà I, 227 Nguyễn Văn Cừ, Q.5, TP.HCM
                    Điện thoại: (028) 62884499
                    Email: info@fit.hcmus.edu.vn"
                    </s>
                    <|user|>
                    Hãy trả lời câu hỏi sau dựa trên ngữ cảnh sau:

                    {context}
                    ---

                    Câu hỏi: {question}
                    </s>
                    <|assistant|>
                    """
        prompt = PromptTemplate(template=query_template, input_variables=["context", "question"])
        return prompt

    def load_rag_pipeline(self, llm, retriever, prompt):
        rag_pipeline = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type='stuff',
            retriever=retriever,
            chain_type_kwargs={
                "prompt": prompt
            },
            return_source_documents=True,
        )
        return rag_pipeline

    def rag(self, source):
        if source == self.current_source:
            return self.rag_pipeline
        else:
            self.retriever = self.load_retriever(retriever_name=source, embeddings=self.embeddings)
            self.rag_pipeline = self.load_rag_pipeline(
                llm=self.pipe,
                retriever=self.retriever,
                prompt=self.prompt
            )
            self.current_source = source
            return self.rag_pipeline


    def calculate_bleu(self, generated_tokens, reference_tokens):
        """
        Calculate BLEU score for a single generated answer and reference.
        :param generated_tokens: List of generated tokens.
        :param reference_tokens: List of reference tokens.
        :return: BLEU score.
        """
        # Use NLTK's BLEU score implementation with smoothing
        smoothie = SmoothingFunction().method4  # Apply smoothing
        bleu_score = sentence_bleu([reference_tokens], generated_tokens, smoothing_function=smoothie)
        return bleu_score

    def calculate_rouge(self, generated_tokens, reference_tokens):
        """
        Calculate ROUGE scores for a single generated answer and reference.
        :param generated_tokens: List of generated tokens.
        :param reference_tokens: List of reference tokens.
        :return: A dictionary with ROUGE-1, ROUGE-2, and ROUGE-L scores.
        """
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(" ".join(reference_tokens), " ".join(generated_tokens))
        return scores


    def evaluate_generation(self, input_filepath: str, output_filepath: str, delay: float = 5.0):
        """
        Evaluate the generated answers using BLEU, ROUGE, METEOR, and SBERT cosine similarity.
        Includes a delay between requests to limit API call frequency.

        :param input_filepath: Path to the input Excel file containing questions and expected outputs.
        :param output_filepath: Path to save the evaluation results as an Excel file.
        :param delay: Time in seconds to wait between processing each question.
        """
        # Load data từ file Excel
        data = pd.read_excel(input_filepath)
        questions = data['question'].tolist()
        expected_outputs = data['expected_output'].tolist()

        results = []

        bleu_scores = []
        rouge_1_scores = []
        rouge_2_scores = []
        rouge_l_scores = []
        meteor_scores = []
        sbert_similarities = []

        for idx, question in enumerate(questions):
            print(f"Processing question {idx + 1}/{len(questions)}: {question}")

            try:
                # Gọi pipeline để lấy câu trả lời
                response = self.rag_pipeline({"query": question})
                generated_answer = response.get("result", "")

                # Tokenize văn bản sử dụng ViTokenizer
                generated_answer_tokens = ViTokenizer.tokenize(generated_answer).split()
                expected_output_tokens = ViTokenizer.tokenize(expected_outputs[idx]).split()

                # Tính BLEU và ROUGE
                bleu_score = self.calculate_bleu(generated_answer_tokens, expected_output_tokens)
                rouge_scores = self.calculate_rouge(generated_answer_tokens, expected_output_tokens)

                # Tính METEOR
                meteor = meteor_score([expected_output_tokens], generated_answer_tokens)

                # Tính SBERT Cosine Similarity
                generated_embedding = self.sbert_model.encode(generated_answer, convert_to_tensor=True)
                expected_embedding = self.sbert_model.encode(expected_outputs[idx], convert_to_tensor=True)
                sbert_similarity = util.pytorch_cos_sim(generated_embedding, expected_embedding).item()

                # Lưu điểm số để tính trung bình
                bleu_scores.append(bleu_score)
                rouge_1_scores.append(rouge_scores["rouge1"].fmeasure)
                rouge_2_scores.append(rouge_scores["rouge2"].fmeasure)
                rouge_l_scores.append(rouge_scores["rougeL"].fmeasure)
                meteor_scores.append(meteor)
                sbert_similarities.append(sbert_similarity)

                # Lưu kết quả
                results.append({
                    "question": question,
                    "generated_answer": generated_answer,
                    "expected_output": expected_outputs[idx],
                    "bleu_score": bleu_score,
                    "rouge_1": rouge_scores["rouge1"].fmeasure,
                    "rouge_2": rouge_scores["rouge2"].fmeasure,
                    "rouge_L": rouge_scores["rougeL"].fmeasure,
                    "meteor": meteor,
                    "sbert_similarity": sbert_similarity
                })
            except Exception as e:
                print(f"Error processing question {idx + 1}: {e}")
                results.append({
                    "question": question,
                    "generated_answer": None,
                    "expected_output": expected_outputs[idx],
                    "bleu_score": None,
                    "rouge_1": None,
                    "rouge_2": None,
                    "rouge_L": None,
                    "meteor": None,
                    "sbert_similarity": None
                })

            # Thêm delay giữa các yêu cầu
            time.sleep(delay)

        # Tính trung bình các độ đo
        avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
        avg_rouge_1 = sum(rouge_1_scores) / len(rouge_1_scores) if rouge_1_scores else 0
        avg_rouge_2 = sum(rouge_2_scores) / len(rouge_2_scores) if rouge_2_scores else 0
        avg_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores) if rouge_l_scores else 0
        avg_meteor = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0
        avg_sbert = sum(sbert_similarities) / len(sbert_similarities) if sbert_similarities else 0

        print("\nAverage Scores:")
        print(f"Average BLEU: {avg_bleu:.4f}")
        print(f"Average ROUGE-1: {avg_rouge_1:.4f}")
        print(f"Average ROUGE-2: {avg_rouge_2:.4f}")
        print(f"Average ROUGE-L: {avg_rouge_l:.4f}")
        print(f"Average METEOR: {avg_meteor:.4f}")
        print(f"Average SBERT Similarity: {avg_sbert:.4f}")

        # Convert results to DataFrame và lưu vào file Excel
        results_df = pd.DataFrame(results)
        results_df.to_excel(output_filepath, index=False)
        print(f"Evaluation results saved to {output_filepath}")


In [19]:
# Có thể thay đổi model khác bằng cách đổi tên model
#Ví dụ:
#model_name = GENERATE_MODEL_NAME
app = LLMServe(model_name="gemini")

In [20]:
input_filepath = "/content/QA_data.xlsx"
output_filepath = "/content/evaluation_results.xlsx"

# Thực hiện đánh giá
app.evaluate_generation(input_filepath, output_filepath=output_filepath)

Processing question 1/144: Tổng số tín chỉ bắt buộc của ngành Hệ thống thông tin là bao nhiêu?
Processing question 2/144: Tên văn bằng ngành Khoa học máy tính sau khi tốt nghiệp là gì?
Processing question 3/144: Sinh viên phải làm gì để đăng ký học phần đầu mỗi học kỳ?
Processing question 4/144: Sinh viên phải học môn nào bắt buộc trong chuyên ngành Khoa học máy tính?
Processing question 5/144: Sinh viên phải hoàn thành bao nhiêu tín chỉ thuộc Kiến thức Giáo dục chuyên nghiệp ngành Công nghệ thông tin?
Processing question 6/144: Sinh viên phải đăng ký tối thiểu bao nhiêu tín chỉ trong học kỳ chính?
Processing question 7/144: Sinh viên có thể kéo dài thời gian học tập tối đa bao lâu?
Processing question 8/144: Sinh viên chuyên ngành Khoa học máy tính cần tích lũy bao nhiêu tín chỉ học phần tự chọn?
Processing question 9/144: Sinh viên cần tích lũy ít nhất bao nhiêu tín chỉ từ học phần bắt buộc của chuyên ngành Khoa học dữ liệu?
Processing question 10/144: Sinh viên cần tích lũy bao nhiê